In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ------------------------------------------------------------
# 🧙‍♂️ StoryWeaver — Chat to Story Generator (Repetition Fix)
# ------------------------------------------------------------
# ✅ Fixed repetitive actions in story generation
# ✅ Enhanced prompts for more cinematic descriptions
# ------------------------------------------------------------

import gradio as gr
from transformers import (
    BartTokenizer, BartForConditionalGeneration,
    AutoTokenizer, AutoModelForSeq2SeqLM
)
import torch
import gc
import os
import re
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 🧼 Environment Setup
# ------------------------------------------------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.set_num_threads(4)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 1️⃣ Load Fine-Tuned BART Summarization Model
# ------------------------------------------------------------
def load_bart_model():
    """Load the fine-tuned BART model for conversation summarization"""
    model_path = "/content/drive/MyDrive/bart_samsum_checkpoints/checkpoint-7366"

    print("Loading BART summarization model...")
    tokenizer = BartTokenizer.from_pretrained(model_path)
    model = BartForConditionalGeneration.from_pretrained(model_path)
    model.eval()
    print("BART model loaded successfully!")
    return tokenizer, model

try:
    bart_tokenizer, bart_model = load_bart_model()
except Exception as e:
    print(f"Error loading BART model: {e}")
    raise

# ------------------------------------------------------------
# 2️⃣ Load Story Generation Model
# ------------------------------------------------------------
def load_story_generator():
    """Load FLAN-T5 for better instruction following"""
    print("Loading FLAN-T5 story generator...")
    model_name = "google/flan-t5-large"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.to("cpu")
    model.eval()
    print("FLAN-T5 model loaded successfully!")
    return tokenizer, model

try:
    story_tokenizer, story_model = load_story_generator()
except Exception as e:
    print(f"Error loading story model: {e}")
    # Fallback to base model
    print("Loading base FLAN-T5 as fallback...")
    story_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    story_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    story_model.to("cpu")
    story_model.eval()

# ------------------------------------------------------------
# 3️⃣ Text Processing Functions
# ------------------------------------------------------------
def clean_chat_text(chat_text):
    """Clean and format chat text"""
    chat_text = re.sub(r'\s+', ' ', chat_text).strip()
    chat_text = re.sub(r':(\S)', r': \1', chat_text)
    return chat_text

def extract_character_names(chat_text):
    """Extract character names from chat"""
    names = re.findall(r'^([A-Za-z]+):', chat_text, re.MULTILINE)
    unique_names = list(set(names))
    return unique_names[:2] if unique_names else ["Emma", "Liam"]

def clean_generated_text(text, prompt_prefixes=None):
    """Clean generated text and remove prompt artifacts"""
    if prompt_prefixes:
        for prefix in prompt_prefixes:
            if text.startswith(prefix):
                text = text[len(prefix):].strip()

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Fix punctuation spacing
    text = re.sub(r'\s*([.!?])\s*', r'\1 ', text)

    # Remove incomplete sentences at the end
    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) > 1 and not sentences[-1].strip().endswith(('.', '!', '?')):
        text = ' '.join(sentences[:-1])

    return text.strip()

def format_summary_coherently(summary):
    """Format summary into a coherent paragraph"""
    # Clean up the text
    summary = re.sub(r'\s+', ' ', summary).strip()

    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', summary)
    sentences = [s.strip() for s in sentences if s.strip()]

    # Remove any fragments that are too short
    sentences = [s for s in sentences if len(s.split()) > 3]

    # Join sentences with proper spacing
    formatted = ' '.join(sentences)

    # Ensure it ends with proper punctuation
    if not formatted.endswith(('.', '!', '?')):
        formatted += '.'

    return formatted

def remove_repetitive_actions(story):
    """Remove repetitive actions and phrases from the story"""
    # Common repetitive phrases to check for
    repetitive_patterns = [
        (r'(\b\w+\b kisses \b\w+\b)(.*?)(\1)', r'\1\2'),  # Repeated kisses
        (r'(\b\w+\b and \b\w+\b kiss)(.*?)(\1)', r'\1\2'),  # Repeated "they kiss"
        (r'(\b\w+\b says,? ["\'].+?["\'])(.*?)(\1)', r'\1\2'),  # Repeated dialogue
        (r'(\b\w+\b \b\w+\b \b\w+\b)(.*?)(\1)', r'\1\2'),  # Repeated three-word actions
    ]

    # Apply each pattern
    for pattern, replacement in repetitive_patterns:
        story = re.sub(pattern, replacement, story, flags=re.IGNORECASE)

    # Remove consecutive very similar sentences
    sentences = re.split(r'(?<=[.!?])\s+', story)
    if len(sentences) > 1:
        cleaned_sentences = [sentences[0]]

        for i in range(1, len(sentences)):
            current = sentences[i].strip()
            previous = cleaned_sentences[-1].strip()

            # Calculate word overlap
            current_words = set(current.lower().split())
            previous_words = set(previous.lower().split())

            if len(current_words) > 0 and len(previous_words) > 0:
                overlap = len(current_words.intersection(previous_words)) / min(len(current_words), len(previous_words))

                # If more than 70% overlap, skip this sentence
                if overlap < 0.7:
                    cleaned_sentences.append(current)
            else:
                cleaned_sentences.append(current)

        story = ' '.join(cleaned_sentences)

    return story

def enhance_story_emotions(story, mood):
    """Enhance the story with more emotional descriptions"""
    # Define mood-specific enhancements
    mood_enhancements = {
        "romance": [
            (r'(\b\w+\b kisses \b\w+\b)', r'\1 gently, their hearts beating as one'),
            (r'(\b\w+\b and \b\w+\b kiss)', r'\1, lost in the moment of their shared affection'),
            (r'(\b\w+\b says,? ["\']I love you["\'])', r'\1, voice filled with deep emotion'),
            (r'(\b\w+\b and \b\w+\b sit)', r'\1 together, enjoying the intimate atmosphere')
        ],
        "drama": [
            (r'(\b\w+\b says,?)', r'\1, voice trembling with emotion'),
            (r'(\b\w+\b and \b\w+\b talk)', r'\1, tension hanging between them'),
            (r'(\b\w+\b looks at \b\w+\b)', r'\1, eyes filled with intensity')
        ],
        "comedy": [
            (r'(\b\w+\b says,?)', r'\1 with a playful grin'),
            (r'(\b\w+\b and \b\w+\b laugh)', r'\1, their laughter filling the room'),
            (r'(\b\w+\b looks at \b\w+\b)', r'\1 with a mischievous twinkle in their eye')
        ],
        "thriller": [
            (r'(\b\w+\b says,?)', r'\1 in a hushed, urgent tone'),
            (r'(\b\w+\b and \b\w+\b look)', r'\1 around cautiously, sensing danger'),
            (r'(\b\w+\b takes \b\w+\b\'s hand)', r'\1 suddenly, gripping tightly with urgency')
        ],
        "inspirational": [
            (r'(\b\w+\b says,?)', r'\1 with determination in their voice'),
            (r'(\b\w+\b and \b\w+\b smile)', r'\1, filled with hope and optimism'),
            (r'(\b\w+\b looks at \b\w+\b)', r'\1 with admiration and respect')
        ]
    }

    # Apply enhancements for the selected mood
    if mood in mood_enhancements:
        for pattern, replacement in mood_enhancements[mood]:
            story = re.sub(pattern, replacement, story, flags=re.IGNORECASE)

    return story

# ------------------------------------------------------------
# 4️⃣ Core Processing Function
# ------------------------------------------------------------
def chat_to_story(chat_text, mood="romance"):
    """Convert chat conversation to summary and story"""
    try:
        # Clean input text
        chat_text = clean_chat_text(chat_text)

        # Validate input
        if not chat_text or len(chat_text.split()) < 10:
            return "Please provide a longer conversation (at least 10 words).", ""

        # Extract character names
        characters = extract_character_names(chat_text)
        char1, char2 = characters[0], characters[1] if len(characters) > 1 else "Liam"

        # --- Step 1: Generate summary with BART using SAMSum format ---
        # Use the SAMSum dataset format for better results
        summary_input = f"dialogue: {chat_text}\nsummary:"

        # Tokenize and truncate
        inputs = bart_tokenizer(
            summary_input,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        # Generate summary with optimized parameters
        with torch.no_grad():
            summary_ids = bart_model.generate(
                **inputs,
                max_length=100,
                min_length=30,
                num_beams=4,
                length_penalty=1.0,
                early_stopping=True,
                repetition_penalty=1.5,
                no_repeat_ngram_size=3
            )

        # Decode and clean summary
        summary = bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        summary = clean_generated_text(summary, ["summary:"])
        summary = format_summary_coherently(summary)

        # --- Step 2: Generate story with FLAN-T5 ---
        # Define mood-specific story elements
        mood_elements = {
            "drama": {
                "emotions": "intense emotions, conflict, and deep feelings",
                "setting": "a tense atmosphere with high stakes",
                "ending": "a resolution that brings emotional catharsis"
            },
            "romance": {
                "emotions": "love, affection, and tender moments",
                "setting": "a romantic atmosphere with intimate details",
                "ending": "a heartwarming conclusion that celebrates their relationship"
            },
            "comedy": {
                "emotions": "humor, lightheartedness, and funny situations",
                "setting": "a playful environment with comedic elements",
                "ending": "a amusing resolution that leaves everyone smiling"
            },
            "thriller": {
                "emotions": "suspense, excitement, and unexpected twists",
                "setting": "a mysterious environment with tension",
                "ending": "a surprising conclusion that resolves the suspense"
            },
            "inspirational": {
                "emotions": "hope, motivation, and uplifting moments",
                "setting": "a positive environment that encourages growth",
                "ending": "an empowering conclusion that inspires"
            }
        }

        # Get mood elements
        elements = mood_elements.get(mood, mood_elements["romance"])

        # Create a detailed, mood-specific prompt with emphasis on variety
        story_prompt = (
            f"Write a {mood} short story (150-250 words) about {char1} and {char2} based on this summary: {summary}. "
            f"The story must include {elements['emotions']}. Set the story in {elements['setting']}. "
            f"Include varied dialogue between {char1} and {char2}, diverse descriptions of their actions and emotions, "
            f"and end with {elements['ending']}. Use varied phrasing and avoid repetitive actions. "
            f"Describe the setting, characters' feelings, and use cinematic language. "
            f"Do not repeat the summary; expand it into a creative narrative with original details."
        )

        # Tokenize prompt
        inputs = story_tokenizer(
            story_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        # Generate story with parameters that discourage repetition
        with torch.no_grad():
            story_ids = story_model.generate(
                **inputs,
                max_length=350,
                min_length=100,
                temperature=0.8,
                top_p=0.9,
                repetition_penalty=2.0,  # Increased penalty
                no_repeat_ngram_size=4,
                do_sample=True,
                num_beams=3
            )

        # Decode and clean story
        story = story_tokenizer.decode(story_ids[0], skip_special_tokens=True)
        story = clean_generated_text(story)

        # Post-process to remove repetitive actions
        story = remove_repetitive_actions(story)

        # Enhance with mood-specific emotional descriptions
        story = enhance_story_emotions(story, mood)

        # Verify story contains character names
        if char1.lower() not in story.lower() or char2.lower() not in story.lower():
            # Try again with more explicit instructions
            explicit_prompt = (
                f"Write a {mood} story about {char1} and {char2}. "
                f"Based on: {summary}. "
                f"Include varied dialogue between {char1} and {char2}. "
                f"Show {elements['emotions']}. "
                f"Set in {elements['setting']}. "
                f"End with {elements['ending']}. "
                f"Use varied descriptions and avoid repetitive actions."
            )

            inputs = story_tokenizer(
                explicit_prompt,
                return_tensors="pt",
                truncation=True,
                max_length=512
            )

            with torch.no_grad():
                story_ids = story_model.generate(
                    **inputs,
                    max_length=350,
                    min_length=100,
                    temperature=0.7,
                    top_p=0.9,
                    repetition_penalty=2.0,
                    no_repeat_ngram_size=4,
                    do_sample=True,
                    num_beams=3
                )

            story = story_tokenizer.decode(story_ids[0], skip_special_tokens=True)
            story = clean_generated_text(story)
            story = remove_repetitive_actions(story)
            story = enhance_story_emotions(story, mood)

        # Format outputs
        formatted_summary = f"🗨️ Chat Summary:\n{summary}"
        formatted_story = f"📖 AI-Generated Story:\n{story}"

        return formatted_summary, formatted_story

    except Exception as e:
        error_msg = f"An error occurred: {str(e)}"
        return error_msg, "Please try again with different input."
    finally:
        # Clean up memory
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ------------------------------------------------------------
# 5️⃣ Gradio Interface Setup
# ------------------------------------------------------------
title = "🧙‍♂️ StoryWeaver — Chat to Story Generator"
description = (
    "Transform your chat conversations into creative stories! "
    "Paste a multi-speaker conversation, select a mood, and watch AI weave a narrative."
)

examples = [
    [
        "Emma: Don't forget our anniversary dinner tonight!\n"
        "Liam: I remember! I made reservations at that new Italian place.\n"
        "Emma: You're the best! I can't wait to try their famous pasta.\n"
        "Liam: It's a surprise, but I heard they have live music too.\n"
        "Emma: Now I'm really excited! See you at 7?",
        "romance"
    ],
    [
        "Alex: Did you finish the report?\n"
        "Jamie: Almost done, but my computer crashed and I lost everything!\n"
        "Alex: No way! How much did you lose?\n"
        "Jamie: All of it. I have to start from scratch.\n"
        "Alex: That's terrible! Can I help in any way?",
        "drama"
    ],
    [
        "Taylor: I think I saw a UFO last night!\n"
        "Morgan: Yeah right! Was it flying or parked?\n"
        "Taylor: Very funny! It had flashing lights and moved silently.\n"
        "Morgan: Probably just a drone or a plane.\n"
        "Taylor: But it zig-zagged across the sky! No plane does that!",
        "thriller"
    ]
]

iface = gr.Interface(
    fn=chat_to_story,
    inputs=[
        gr.Textbox(
            label="💬 Paste your chat conversation here:",
            lines=12,
            placeholder="e.g. Emma: Let's meet at the café tomorrow.\nLiam: Sure! ☕",
            info="Include speaker names (e.g., Emma: Hello!)"
        ),
        gr.Dropdown(
            ["drama", "romance", "comedy", "thriller", "inspirational"],
            label="🎭 Choose a mood",
            value="romance",
            info="Select the emotional tone for your story"
        )
    ],
    outputs=[
        gr.Textbox(
            label="🗨️ Chat Summary",
            lines=6,
            info="Summary of your conversation",
            elem_classes=["scrollable"]
        ),
        gr.Textbox(
            label="📖 AI-Generated Story",
            lines=12,
            info="150-250 word story based on your chat and selected mood",
            elem_classes=["scrollable"]
        )
    ],
    title=title,
    description=description,
    examples=examples,
    theme="soft",
    allow_flagging="never",
    css="""
    .scrollable {
        overflow-y: auto !important;
        max-height: 300px;
    }
    .gradio-container {
        max-width: 1000px !important;
        margin: 0 auto;
    }
    """
)

# ------------------------------------------------------------
# 6️⃣ Launch the App
# ------------------------------------------------------------
if __name__ == "__main__":
    print("Launching StoryWeaver app...")
    iface.launch(share=True)

Loading BART summarization model...
BART model loaded successfully!
Loading FLAN-T5 story generator...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded successfully!
Launching StoryWeaver app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a66c77b6b205183407.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
